In [1]:
# ============================================================
# FIND + QC + MERGE WATER BALANCE FILES: 1981–2025
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import xarray as xr

# Change this only if your project folder is somewhere else
water_balance_root = r"C:\Users\franc\Water Balance-1981-2025"

# Find all yearly water-balance NetCDF files recursively
files = sorted(
    glob.glob(
        os.path.join(
            water_balance_root,
            "**",
            "water_balance_*.nc"
        ),
        recursive=True
    )
)

print("Number of water-balance files found:", len(files))

for f in files[:5]:
    print(f)

print("...")

for f in files[-5:]:
    print(f)

Number of water-balance files found: 45
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_1981.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_1982.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_1983.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_1984.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_1985.nc
...
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_2021.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_2022.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_2023.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_2024.nc
C:\Users\franc\Water Balance-1981-2025\Water Balance\water_balance_2025.nc


In [2]:
# ============================================================
# EXPECTED YEARS
# ============================================================

expected_years = list(range(1981, 2026))

yearly_arrays = []
found_years = []

reference_lat = None
reference_lon = None

print("\n" + "=" * 70)
print("YEAR-BY-YEAR QUALITY CONTROL")
print("=" * 70)

for f in files:

    filename = os.path.basename(f)

    # Extract year
    year = int(
        filename
        .replace("water_balance_", "")
        .replace(".nc", "")
    )

    found_years.append(year)

    ds = xr.open_dataset(f)

    # Get water-balance variable
    if "water_balance" in ds.data_vars:
        wb = ds["water_balance"]
    elif len(ds.data_vars) == 1:
        wb = ds[list(ds.data_vars)[0]]
    else:
        raise ValueError(
            f"{year}: cannot identify water-balance variable."
        )

    # Leap-year handling
    is_leap = (
        year % 4 == 0
        and (year % 100 != 0 or year % 400 == 0)
    )

    expected_days = 366 if is_leap else 365

    print(f"\n{year}")
    print("-" * 40)

    print("Shape:", wb.shape)
    print("Days:", wb.sizes["time"], "/", expected_days)
    print(
        "Dates:",
        str(wb.time.values[0])[:10],
        "to",
        str(wb.time.values[-1])[:10]
    )

    # Check day count
    if wb.sizes["time"] != expected_days:
        raise ValueError(
            f"{year}: expected {expected_days} days, "
            f"found {wb.sizes['time']}"
        )

    # Check duplicate dates
    if len(np.unique(wb.time.values)) != expected_days:
        raise ValueError(
            f"{year}: duplicate dates detected."
        )

    # Check missing calendar dates
    expected_dates = pd.date_range(
        f"{year}-01-01",
        f"{year}-12-31",
        freq="D"
    )

    actual_dates = pd.DatetimeIndex(
        wb.time.values
    )

    missing_dates = expected_dates.difference(
        actual_dates
    )

    if len(missing_dates) > 0:
        print("Missing dates:", list(missing_dates))
        raise ValueError(
            f"{year}: missing dates detected."
        )

    # Check spatial grid
    if reference_lat is None:

        reference_lat = wb.lat.values.copy()
        reference_lon = wb.lon.values.copy()

    else:

        if not np.allclose(
            wb.lat.values,
            reference_lat,
            atol=1e-6
        ):
            raise ValueError(
                f"{year}: latitude grid mismatch."
            )

        if not np.allclose(
            wb.lon.values,
            reference_lon,
            atol=1e-6
        ):
            raise ValueError(
                f"{year}: longitude grid mismatch."
            )

    print("Grid:", wb.sizes["lat"], "x", wb.sizes["lon"])

    print(
        "Minimum:",
        float(wb.min(skipna=True).values)
    )

    print(
        "Maximum:",
        float(wb.max(skipna=True).values)
    )

    print(
        "Mean:",
        float(wb.mean(skipna=True).values)
    )

    yearly_arrays.append(wb)


# ============================================================
# CHECK ALL YEARS FOUND
# ============================================================

missing_years = sorted(
    set(expected_years) - set(found_years)
)

print("\nMissing years:", missing_years)

if missing_years:
    raise ValueError(
        f"Missing years: {missing_years}"
    )


# ============================================================
# ============================================================
# STANDARDISE GRID + MERGE 1981–2025
# ============================================================

print("\n" + "=" * 70)
print("STANDARDISING GRID")
print("=" * 70)

# Use the first year's coordinates as the master grid
master_lat = yearly_arrays[0].lat.values.copy()
master_lon = yearly_arrays[0].lon.values.copy()

yearly_arrays_fixed = []

for wb in yearly_arrays:

    # We already confirmed during QC that all grids are
    # numerically equivalent, so simply assign identical
    # coordinate labels.
    wb_fixed = wb.assign_coords(
        lat=master_lat,
        lon=master_lon
    )

    yearly_arrays_fixed.append(wb_fixed)

print("All yearly grids assigned identical coordinates.")


# ============================================================
# MERGE
# ============================================================

print("\n" + "=" * 70)
print("MERGING 1981–2025")
print("=" * 70)

water_balance_all = xr.concat(
    yearly_arrays_fixed,
    dim="time",
    join="exact"
)

water_balance_all = water_balance_all.sortby("time")

water_balance_all.name = "water_balance"

water_balance_all.attrs["units"] = "mm/day"
water_balance_all.attrs["long_name"] = (
    "Daily climatic water balance (P - Ep)"
)

print("Merge completed successfully!")


YEAR-BY-YEAR QUALITY CONTROL

1981
----------------------------------------
Shape: (365, 141, 220)
Days: 365 / 365
Dates: 1981-01-01 to 1981-12-31
Grid: 141 x 220
Minimum: -13.793437004089355
Maximum: 164.4240567363437
Mean: -3.4442935687402327

1982
----------------------------------------
Shape: (365, 141, 220)
Days: 365 / 365
Dates: 1982-01-01 to 1982-12-31
Grid: 141 x 220
Minimum: -14.156990051190542
Maximum: 117.52663228543061
Mean: -3.616289675401237

1983
----------------------------------------
Shape: (365, 141, 220)
Days: 365 / 365
Dates: 1983-01-01 to 1983-12-31
Grid: 141 x 220
Minimum: -15.28117465969946
Maximum: 140.78391896205855
Mean: -3.6355692653805605

1984
----------------------------------------
Shape: (366, 141, 220)
Days: 366 / 366
Dates: 1984-01-01 to 1984-12-31
Grid: 141 x 220
Minimum: -13.906404495204852
Maximum: 213.40552847687664
Mean: -3.656455642488498

1985
----------------------------------------
Shape: (365, 141, 220)
Days: 365 / 365
Dates: 1985-01-01 to

In [3]:
# ============================================================
# FINAL CONTINUITY CHECK
# ============================================================

expected_full_time = pd.date_range(
    start="1981-01-01",
    end="2025-12-31",
    freq="D"
)

actual_full_time = pd.DatetimeIndex(
    water_balance_all.time.values
)

missing_dates = expected_full_time.difference(
    actual_full_time
)

extra_dates = actual_full_time.difference(
    expected_full_time
)

duplicate_dates = (
    water_balance_all.sizes["time"]
    - len(np.unique(water_balance_all.time.values))
)

print("\n" + "=" * 70)
print("FINAL WATER BALANCE CHECK: 1981–2025")
print("=" * 70)

print("Shape:", water_balance_all.shape)

print(
    "First date:",
    water_balance_all.time.values[0]
)

print(
    "Last date:",
    water_balance_all.time.values[-1]
)

print(
    "Total days:",
    water_balance_all.sizes["time"]
)

print(
    "Expected days:",
    len(expected_full_time)
)

print(
    "Missing dates:",
    len(missing_dates)
)

print(
    "Extra dates:",
    len(extra_dates)
)

print(
    "Duplicate dates:",
    duplicate_dates
)

print(
    "Latitude:",
    water_balance_all.sizes["lat"]
)

print(
    "Longitude:",
    water_balance_all.sizes["lon"]
)


FINAL WATER BALANCE CHECK: 1981–2025
Shape: (16436, 141, 220)
First date: 1981-01-01T00:00:00.000000000
Last date: 2025-12-31T00:00:00.000000000
Total days: 16436
Expected days: 16436
Missing dates: 0
Extra dates: 0
Duplicate dates: 0
Latitude: 141
Longitude: 220


In [4]:
output_file = r"C:\Users\franc\water_balance_merged_1981_2025.nc"

water_balance_all.to_netcdf(output_file)

print("Saved successfully:")
print(output_file)

Saved successfully:
C:\Users\franc\water_balance_merged_1981_2025.nc


In [6]:
# ============================================================
# STAGE 1: CONTINUOUS WATER BALANCE -> 30-DAY / 90-DAY
# ============================================================

import xarray as xr
import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# 1. FILE PATH
# ------------------------------------------------------------

input_file = r"C:\Users\franc\water_balance_merged_1981_2025.nc"

print("File exists:", os.path.exists(input_file))

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"File not found:\n{input_file}"
    )

# ------------------------------------------------------------
# 2. OPEN MERGED WATER BALANCE
# ------------------------------------------------------------

ds = xr.open_dataset(input_file)

print("\nDataset opened successfully.")

print("Variables:", list(ds.data_vars))
print("Dimensions:", dict(ds.sizes))
print("Coordinates:", list(ds.coords))

# ------------------------------------------------------------
# 3. GET WATER BALANCE
# ------------------------------------------------------------

if "water_balance" not in ds.data_vars:
    raise ValueError(
        f"'water_balance' variable not found.\n"
        f"Available variables: {list(ds.data_vars)}"
    )

wb = ds["water_balance"]

print("\nWater balance information")
print("-------------------------")

print("Shape:", wb.shape)
print("Dimensions:", wb.dims)
print("First date:", wb.time.values[0])
print("Last date:", wb.time.values[-1])
print("Latitude:", wb.sizes["lat"])
print("Longitude:", wb.sizes["lon"])
# ------------------------------------------------------------
# 4. CHECK CONTINUITY AGAIN
# ------------------------------------------------------------

expected_dates = pd.date_range(
    "1981-01-01",
    "2025-12-31",
    freq="D"
)

actual_dates = pd.DatetimeIndex(
    wb.time.values
)

print("\nTime check")
print("-------------------------")
print("Expected days:", len(expected_dates))
print("Actual days:", wb.sizes["time"])
print("Unique dates:", len(np.unique(wb.time.values)))
print("Missing dates:", len(expected_dates.difference(actual_dates)))

if wb.sizes["time"] != len(expected_dates):
    raise ValueError("Unexpected number of daily observations.")

if len(np.unique(wb.time.values)) != wb.sizes["time"]:
    raise ValueError("Duplicate dates detected.")

if len(expected_dates.difference(actual_dates)) != 0:
    raise ValueError("Missing dates detected.")

print("Continuous time series confirmed!")

# ------------------------------------------------------------
# 5. 30-DAY ACCUMULATED WATER BALANCE
# SPEI-1 input
# ------------------------------------------------------------

print("\nCalculating 30-day accumulated water balance...")

wb_30 = wb.rolling(
    time=30,
    min_periods=30
).sum()

wb_30.name = "water_balance_30d"

wb_30.attrs["units"] = "mm"
wb_30.attrs["long_name"] = (
    "30-day accumulated climatic water balance"
)

# ------------------------------------------------------------
# 6. 90-DAY ACCUMULATED WATER BALANCE
# SPEI-3 input
# ------------------------------------------------------------

print("Calculating 90-day accumulated water balance...")

wb_90 = wb.rolling(
    time=90,
    min_periods=90
).sum()

wb_90.name = "water_balance_90d"

wb_90.attrs["units"] = "mm"
wb_90.attrs["long_name"] = (
    "90-day accumulated climatic water balance"
)

# ------------------------------------------------------------
# 7. BASIC CHECK
# ------------------------------------------------------------

print("\n30-day water balance")
print("-------------------------")
print("Shape:", wb_30.shape)
print("First valid date:",
      wb_30.dropna("time", how="all").time.values[0])

print("\n90-day water balance")
print("-------------------------")
print("Shape:", wb_90.shape)
print("First valid date:",
      wb_90.dropna("time", how="all").time.values[0])

# ------------------------------------------------------------
# 8. TEST LOCATION
# ------------------------------------------------------------

lat_test = -27.95
lon_test = 21.05

wb30_test = wb_30.sel(
    lat=lat_test,
    lon=lon_test,
    method="nearest"
)

wb90_test = wb_90.sel(
    lat=lat_test,
    lon=lon_test,
    method="nearest"
)

print("\nTest location")
print("-------------------------")
print("Latitude:", float(wb30_test.lat))
print("Longitude:", float(wb30_test.lon))

print("\nFirst 5 valid 30-day values:")
print(
    wb30_test.dropna("time").values[:5]
)

print("\nFirst 5 valid 90-day values:")
print(
    wb90_test.dropna("time").values[:5]
)

File exists: True

Dataset opened successfully.
Variables: ['water_balance']
Dimensions: {'time': 16436, 'lat': 141, 'lon': 220}
Coordinates: ['time', 'lat', 'lon']

Water balance information
-------------------------
Shape: (16436, 141, 220)
Dimensions: ('time', 'lat', 'lon')
First date: 1981-01-01T00:00:00.000000000
Last date: 2025-12-31T00:00:00.000000000
Latitude: 141
Longitude: 220

Time check
-------------------------
Expected days: 16436
Actual days: 16436
Unique dates: 16436
Missing dates: 0
Continuous time series confirmed!

Calculating 30-day accumulated water balance...
Calculating 90-day accumulated water balance...

30-day water balance
-------------------------
Shape: (16436, 141, 220)
First valid date: 1981-01-30T00:00:00.000000000

90-day water balance
-------------------------
Shape: (16436, 141, 220)
First valid date: 1981-03-31T00:00:00.000000000

Test location
-------------------------
Latitude: -27.950000000003527
Longitude: 21.049999999988586

First 5 valid 30-day

In [10]:
import gc

for name in [
    "spei1",
    "spei3",
    "wb30_mean",
    "wb30_std",
    "wb90_mean",
    "wb90_std"
]:
    if name in globals():
        del globals()[name]

gc.collect()

print("Memory cleaned.")

Memory cleaned.


In [11]:
# ============================================================
# TEST SPEI STANDARDISATION AT ONE LOCATION
# ============================================================

lat_test = -27.95
lon_test = 21.05

test30 = wb_30.sel(
    lat=lat_test,
    lon=lon_test,
    method="nearest"
).astype("float32")

test90 = wb_90.sel(
    lat=lat_test,
    lon=lon_test,
    method="nearest"
).astype("float32")


def standardize_by_dayofyear(da):

    climatology_mean = (
        da.groupby("time.dayofyear")
        .mean("time")
    )

    climatology_std = (
        da.groupby("time.dayofyear")
        .std("time")
    )

    anomaly = (
        da.groupby("time.dayofyear")
        - climatology_mean
    )

    # IMPORTANT:
    # regroup the anomaly before division
    # so dayofyear is matched correctly
    standardized = (
        anomaly.groupby("time.dayofyear")
        / climatology_std
    )

    return standardized.astype("float32")


spei1_test = standardize_by_dayofyear(test30)
spei3_test = standardize_by_dayofyear(test90)


print("SPEI-1")
print("-------------------------")
print("Shape:", spei1_test.shape)
print("Minimum:", float(spei1_test.min(skipna=True)))
print("Maximum:", float(spei1_test.max(skipna=True)))
print("Mean:", float(spei1_test.mean(skipna=True)))

print("\nFirst 10 valid values:")
print(
    spei1_test.dropna("time").values[:10]
)


print("\nSPEI-3")
print("-------------------------")
print("Shape:", spei3_test.shape)
print("Minimum:", float(spei3_test.min(skipna=True)))
print("Maximum:", float(spei3_test.max(skipna=True)))
print("Mean:", float(spei3_test.mean(skipna=True)))

print("\nFirst 10 valid values:")
print(
    spei3_test.dropna("time").values[:10]
)

SPEI-1
-------------------------
Shape: (16436,)
Minimum: -3.1164441108703613
Maximum: 3.406891345977783
Mean: 5.932055273660808e-07

First 10 valid values:
[-1.0850203  -1.0851034  -1.1334834  -1.0893021  -1.0752692  -1.0947852
 -1.0964999  -1.1114981  -0.78409606 -0.77876997]

SPEI-3
-------------------------
Shape: (16436,)
Minimum: -2.8551716804504395
Maximum: 2.7874579429626465
Mean: 1.832367928500389e-07

First 10 valid values:
[-0.7378689  -0.75952804 -0.7777173  -0.76713127 -0.75512135 -0.75951654
 -0.8045536  -0.80238676 -0.79182655 -0.7953836 ]


In [12]:
# ============================================================
# FULL-GRID SPEI-LIKE CALCULATION IN LATITUDE CHUNKS
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

block_size = 5

output_file = r"C:\Users\franc\spei1_spei3_1981_2025.nc"

# ------------------------------------------------------------
# STANDARDISATION FUNCTION
# ------------------------------------------------------------

def standardize_by_dayofyear(da):

    clim_mean = (
        da.groupby("time.dayofyear")
        .mean("time")
    )

    clim_std = (
        da.groupby("time.dayofyear")
        .std("time")
    )

    anomaly = (
        da.groupby("time.dayofyear")
        - clim_mean
    )

    standardized = (
        anomaly.groupby("time.dayofyear")
        / clim_std
    )

    return standardized.astype("float32")


# ============================================================
# PREPARE OUTPUT LISTS
# ============================================================

spei1_blocks = []
spei3_blocks = []

nlat = wb_30.sizes["lat"]

print("Total latitude rows:", nlat)
print("Block size:", block_size)


# ============================================================
# PROCESS BLOCK BY BLOCK
# ============================================================

for start in range(0, nlat, block_size):

    end = min(
        start + block_size,
        nlat
    )

    print(
        f"\nProcessing latitude rows "
        f"{start}:{end} of {nlat}"
    )

    # --------------------------------------------------------
    # Extract only a small latitude block
    # --------------------------------------------------------

    wb30_block = wb_30.isel(
        lat=slice(start, end)
    ).astype("float32")

    wb90_block = wb_90.isel(
        lat=slice(start, end)
    ).astype("float32")

    # --------------------------------------------------------
    # Calculate SPEI-like indices
    # --------------------------------------------------------

    spei1_block = standardize_by_dayofyear(
        wb30_block
    )

    spei3_block = standardize_by_dayofyear(
        wb90_block
    )

    # --------------------------------------------------------
    # Force computation while block is small
    # --------------------------------------------------------

    spei1_block = spei1_block.load()
    spei3_block = spei3_block.load()

    # --------------------------------------------------------
    # Keep completed blocks
    # --------------------------------------------------------

    spei1_blocks.append(
        spei1_block
    )

    spei3_blocks.append(
        spei3_block
    )

    # --------------------------------------------------------
    # Clean temporary objects
    # --------------------------------------------------------

    del wb30_block
    del wb90_block
    del spei1_block
    del spei3_block

    gc.collect()


# ============================================================
# COMBINE LATITUDE BLOCKS
# ============================================================

print("\nCombining SPEI-1 blocks...")

spei1 = xr.concat(
    spei1_blocks,
    dim="lat"
)

print("Combining SPEI-3 blocks...")

spei3 = xr.concat(
    spei3_blocks,
    dim="lat"
)


# ============================================================
# CLEAN UP BLOCK LISTS
# ============================================================

del spei1_blocks
del spei3_blocks

gc.collect()


# ============================================================
# FINAL VARIABLE INFORMATION
# ============================================================

spei1.name = "spei1"

spei1.attrs["long_name"] = (
    "Standardized 30-day climatic water balance index"
)

spei1.attrs["description"] = (
    "SPEI-like index based on day-of-year standardization "
    "of 30-day accumulated P-Ep"
)


spei3.name = "spei3"

spei3.attrs["long_name"] = (
    "Standardized 90-day climatic water balance index"
)

spei3.attrs["description"] = (
    "SPEI-like index based on day-of-year standardization "
    "of 90-day accumulated P-Ep"
)


# ============================================================
# CHECK SHAPES
# ============================================================

print("\n" + "=" * 70)
print("FULL SPEI CHECK")
print("=" * 70)

print("SPEI-1 shape:", spei1.shape)
print("SPEI-3 shape:", spei3.shape)

print(
    "First date:",
    spei1.time.values[0]
)

print(
    "Last date:",
    spei1.time.values[-1]
)

print(
    "Latitude:",
    spei1.sizes["lat"]
)

print(
    "Longitude:",
    spei1.sizes["lon"]
)


# ============================================================
# TEST LOCATION AGAIN
# ============================================================

lat_test = -27.95
lon_test = 21.05

spei1_test = spei1.sel(
    lat=lat_test,
    lon=lon_test,
    method="nearest"
)

spei3_test = spei3.sel(
    lat=lat_test,
    lon=lon_test,
    method="nearest"
)

print("\nTest location")
print("-" * 40)

print("SPEI-1 mean:",
      float(spei1_test.mean(skipna=True).values))

print("SPEI-1 min:",
      float(spei1_test.min(skipna=True).values))

print("SPEI-1 max:",
      float(spei1_test.max(skipna=True).values))

print("\nSPEI-3 mean:",
      float(spei3_test.mean(skipna=True).values))

print("SPEI-3 min:",
      float(spei3_test.min(skipna=True).values))

print("SPEI-3 max:",
      float(spei3_test.max(skipna=True).values))


# ============================================================
# SAVE BOTH INDICES
# ============================================================

print("\nSaving SPEI dataset...")

spei_ds = xr.Dataset(
    {
        "spei1": spei1,
        "spei3": spei3
    }
)

spei_ds.to_netcdf(
    output_file
)

print("\nSaved successfully:")
print(output_file)

print("\nDONE!")

Total latitude rows: 141
Block size: 5

Processing latitude rows 0:5 of 141

Processing latitude rows 5:10 of 141

Processing latitude rows 10:15 of 141

Processing latitude rows 15:20 of 141

Processing latitude rows 20:25 of 141

Processing latitude rows 25:30 of 141

Processing latitude rows 30:35 of 141

Processing latitude rows 35:40 of 141

Processing latitude rows 40:45 of 141

Processing latitude rows 45:50 of 141

Processing latitude rows 50:55 of 141

Processing latitude rows 55:60 of 141

Processing latitude rows 60:65 of 141

Processing latitude rows 65:70 of 141

Processing latitude rows 70:75 of 141

Processing latitude rows 75:80 of 141

Processing latitude rows 80:85 of 141

Processing latitude rows 85:90 of 141

Processing latitude rows 90:95 of 141

Processing latitude rows 95:100 of 141

Processing latitude rows 100:105 of 141

Processing latitude rows 105:110 of 141

Processing latitude rows 110:115 of 141

Processing latitude rows 115:120 of 141

Processing latitud

In [13]:
# ============================================================
# STAGE 3: FLASH DROUGHT DETECTION
# 1981–2025 SPEI-1 / SPEI-3
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

# ------------------------------------------------------------
# 1. INPUT / OUTPUT
# ------------------------------------------------------------

input_file = r"C:\Users\franc\spei1_spei3_1981_2025.nc"

output_file = r"C:\Users\franc\flash_drought_metrics_1982_2025.nc"

block_size = 3


# ------------------------------------------------------------
# 2. OPEN SPEI DATASET
# ------------------------------------------------------------

ds = xr.open_dataset(input_file)

print("=" * 70)
print("FLASH DROUGHT ANALYSIS")
print("=" * 70)

print("\nVariables:")
print(list(ds.data_vars))

print("\nDimensions:")
print(dict(ds.sizes))

print("\nFirst date:", ds.time.values[0])
print("Last date :", ds.time.values[-1])


# ------------------------------------------------------------
# 3. CHECK VARIABLES
# ------------------------------------------------------------

if "spei1" not in ds.data_vars:
    raise ValueError("spei1 not found.")

if "spei3" not in ds.data_vars:
    raise ValueError("spei3 not found.")


# ============================================================
# IMPORTANT THRESHOLDS FROM SUPERVISOR CODE
# ============================================================

RAPID_DECLINE_THRESHOLD = -1.0
SPEI1_DROUGHT_THRESHOLD = -1.2
SPEI3_ACTIVE_THRESHOLD = -1.0
DECLINE_WINDOW = 20


# ------------------------------------------------------------
# 4. OUTPUT BLOCK LISTS
# ------------------------------------------------------------

frequency_blocks = []
duration_blocks = []
intensity_blocks = []

nlat = ds.sizes["lat"]

print("\nLatitude rows:", nlat)
print("Block size:", block_size)


# ============================================================
# 5. PROCESS LATITUDE BLOCKS
# ============================================================

for start in range(0, nlat, block_size):

    end = min(
        start + block_size,
        nlat
    )

    print(
        f"\nProcessing latitude rows "
        f"{start}:{end} of {nlat}"
    )

    # --------------------------------------------------------
    # LOAD SMALL BLOCK
    # --------------------------------------------------------

    spei1_block = (
        ds["spei1"]
        .isel(lat=slice(start, end))
        .load()
        .astype("float32")
    )

    spei3_block = (
        ds["spei3"]
        .isel(lat=slice(start, end))
        .load()
        .astype("float32")
    )


    # ========================================================
    # 6. TRUE 20-DAY SPEI-1 CHANGE
    # ========================================================
    #
    # Current day minus value 20 calendar days earlier.
    #
    # Example:
    # SPEI today = -1.5
    # SPEI 20 days ago = -0.2
    #
    # Change = -1.3
    #
    # This satisfies the <= -1.0 rapid-decline criterion.
    # ========================================================

    spei1_change_20d = (
        spei1_block
        -
        spei1_block.shift(time=DECLINE_WINDOW)
    )


    # ========================================================
    # 7. FLASH DROUGHT ONSET
    # ========================================================

    fd_onset = xr.where(
        (
            spei1_change_20d
            <= RAPID_DECLINE_THRESHOLD
        )
        &
        (
            spei1_block
            <= SPEI1_DROUGHT_THRESHOLD
        ),
        1,
        0
    ).astype("int8")


    # ========================================================
    # 8. ACTIVE DROUGHT STATE
    # ========================================================

    fd_active = xr.where(
        spei3_block <= SPEI3_ACTIVE_THRESHOLD,
        1,
        0
    ).astype("int8")


    # ========================================================
    # 9. OCTOBER–APRIL MAIZE SEASON
    # ========================================================

    growing_mask = (
        spei1_block.time.dt.month.isin(
            [10, 11, 12, 1, 2, 3, 4]
        )
    )

    fd_onset_growing = fd_onset.sel(
        time=growing_mask
    )

    fd_active_growing = fd_active.sel(
        time=growing_mask
    )

    spei3_growing = spei3_block.sel(
        time=growing_mask
    )


    # ========================================================
    # 10. ASSIGN HARVEST YEAR
    # ========================================================
    #
    # Oct–Dec 1981 -> harvest year 1982
    # Jan–Apr 1982 -> harvest year 1982
    # ========================================================

    harvest_year = (
        fd_onset_growing.time.dt.year
        +
        xr.where(
            fd_onset_growing.time.dt.month >= 10,
            1,
            0
        )
    )

    fd_onset_growing = (
        fd_onset_growing.assign_coords(
            harvest_year=("time", harvest_year.values)
        )
    )

    fd_active_growing = (
        fd_active_growing.assign_coords(
            harvest_year=("time", harvest_year.values)
        )
    )

    spei3_growing = (
        spei3_growing.assign_coords(
            harvest_year=("time", harvest_year.values)
        )
    )


    # ========================================================
    # 11. KEEP COMPLETE GROWING SEASONS ONLY
    # ========================================================
    #
    # 1981 harvest year is incomplete because we don't have
    # Oct-Dec 1980.
    #
    # 2026 is incomplete because our data end in Dec 2025.
    #
    # Therefore complete Oct-Apr seasons are:
    # 1982 through 2025.
    # ========================================================

    complete_seasons = (
        (fd_onset_growing.harvest_year >= 1982)
        &
        (fd_onset_growing.harvest_year <= 2025)
    )

    fd_onset_growing = fd_onset_growing.sel(
        time=complete_seasons
    )

    fd_active_growing = fd_active_growing.sel(
        time=complete_seasons
    )

    spei3_growing = spei3_growing.sel(
        time=complete_seasons
    )


    # ========================================================
    # 12. FLASH DROUGHT FREQUENCY
    # ========================================================
    #
    # Baseline implementation following the supplied script:
    # number of onset-trigger days per harvest year.
    # ========================================================

    fd_frequency = (
        fd_onset_growing
        .groupby("harvest_year")
        .sum("time")
        .astype("int16")
    )


    # ========================================================
    # 13. DURATION
    # ========================================================
    #
    # Number of growing-season days with SPEI-3 <= -1.0
    # ========================================================

    fd_duration = (
        fd_active_growing
        .groupby("harvest_year")
        .sum("time")
        .astype("int16")
    )


    # ========================================================
    # 14. INTENSITY
    # ========================================================
    #
    # Accumulated magnitude of negative SPEI-3 on active days.
    # ========================================================

    fd_intensity = (
        (-spei3_growing)
        .where(
            fd_active_growing == 1,
            0
        )
        .groupby("harvest_year")
        .sum("time")
        .astype("float32")
    )


    # --------------------------------------------------------
    # LOAD SMALL RESULTS
    # --------------------------------------------------------

    fd_frequency = fd_frequency.load()
    fd_duration = fd_duration.load()
    fd_intensity = fd_intensity.load()


    frequency_blocks.append(
        fd_frequency
    )

    duration_blocks.append(
        fd_duration
    )

    intensity_blocks.append(
        fd_intensity
    )


    # --------------------------------------------------------
    # CLEAN MEMORY
    # --------------------------------------------------------

    del spei1_block
    del spei3_block
    del spei1_change_20d

    del fd_onset
    del fd_active

    del fd_onset_growing
    del fd_active_growing
    del spei3_growing

    del fd_frequency
    del fd_duration
    del fd_intensity

    gc.collect()


# ============================================================
# 15. COMBINE LATITUDE BLOCKS
# ============================================================

print("\nCombining frequency blocks...")

fd_frequency = xr.concat(
    frequency_blocks,
    dim="lat"
)

print("Combining duration blocks...")

fd_duration = xr.concat(
    duration_blocks,
    dim="lat"
)

print("Combining intensity blocks...")

fd_intensity = xr.concat(
    intensity_blocks,
    dim="lat"
)


del frequency_blocks
del duration_blocks
del intensity_blocks

gc.collect()


# ============================================================
# 16. CREATE OUTPUT DATASET
# ============================================================

flash_drought = xr.Dataset(
    {
        "fd_frequency": fd_frequency,
        "fd_duration": fd_duration,
        "fd_intensity": fd_intensity
    }
)


flash_drought["fd_frequency"].attrs[
    "long_name"
] = "Flash drought onset trigger frequency"

flash_drought["fd_duration"].attrs[
    "long_name"
] = "Days with sustained SPEI-3 drought conditions"

flash_drought["fd_duration"].attrs[
    "units"
] = "days"

flash_drought["fd_intensity"].attrs[
    "long_name"
] = "Cumulative SPEI-3 drought intensity"


# ============================================================
# 17. FINAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("FLASH DROUGHT METRICS")
print("=" * 70)

print("Dimensions:")
print(dict(flash_drought.sizes))

print(
    "\nFirst harvest year:",
    int(flash_drought.harvest_year.values[0])
)

print(
    "Last harvest year:",
    int(flash_drought.harvest_year.values[-1])
)

print(
    "Number of harvest years:",
    flash_drought.sizes["harvest_year"]
)

print(
    "\nMaximum frequency:",
    int(
        flash_drought[
            "fd_frequency"
        ].max().values
    )
)

print(
    "Maximum duration:",
    int(
        flash_drought[
            "fd_duration"
        ].max().values
    )
)

print(
    "Maximum intensity:",
    float(
        flash_drought[
            "fd_intensity"
        ].max().values
    )
)


# ============================================================
# 18. TEST LOCATION
# ============================================================

test = flash_drought.sel(
    lat=-27.95,
    lon=21.05,
    method="nearest"
)

print("\nTest location")
print("-" * 40)

print(
    "Latitude:",
    float(test.lat)
)

print(
    "Longitude:",
    float(test.lon)
)

print("\nFirst 10 harvest years:")
print(
    test.harvest_year.values[:10]
)

print("\nFirst 10 frequencies:")
print(
    test.fd_frequency.values[:10]
)

print("\nFirst 10 durations:")
print(
    test.fd_duration.values[:10]
)

print("\nFirst 10 intensities:")
print(
    test.fd_intensity.values[:10]
)


# ============================================================
# 19. SAVE
# ============================================================

print("\nSaving flash drought metrics...")

flash_drought.to_netcdf(
    output_file
)

print("\nSaved successfully:")
print(output_file)

print("\nDONE!")

FLASH DROUGHT ANALYSIS

Variables:
['spei1', 'spei3']

Dimensions:
{'time': 16436, 'lat': 141, 'lon': 220}

First date: 1981-01-01T00:00:00.000000000
Last date : 2025-12-31T00:00:00.000000000

Latitude rows: 141
Block size: 3

Processing latitude rows 0:3 of 141

Processing latitude rows 3:6 of 141

Processing latitude rows 6:9 of 141

Processing latitude rows 9:12 of 141

Processing latitude rows 12:15 of 141

Processing latitude rows 15:18 of 141

Processing latitude rows 18:21 of 141

Processing latitude rows 21:24 of 141

Processing latitude rows 24:27 of 141

Processing latitude rows 27:30 of 141

Processing latitude rows 30:33 of 141

Processing latitude rows 33:36 of 141

Processing latitude rows 36:39 of 141

Processing latitude rows 39:42 of 141

Processing latitude rows 42:45 of 141

Processing latitude rows 45:48 of 141

Processing latitude rows 48:51 of 141

Processing latitude rows 51:54 of 141

Processing latitude rows 54:57 of 141

Processing latitude rows 57:60 of 141



In [14]:
# ============================================================
# REFINED FLASH DROUGHT EVENT TRACKING
# Supervisor clarification:
#
# Onset:
#   20-day SPEI-1 decline <= -1.0
#   AND SPEI-1 <= -1.2
#
# Drought state:
#   SPEI-3 <= -1.0
#
# Termination:
#   SPEI-3 > -0.5
#
# Intermediate recovery:
#   -1.0 < SPEI-3 <= -0.5
#   remains part of the same event
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

# ============================================================
# 1. PATHS
# ============================================================

input_file = r"C:\Users\franc\spei1_spei3_1981_2025.nc"

output_file = (
    r"C:\Users\franc\flash_drought_events_1982_2025.nc"
)

block_size = 3


# ============================================================
# 2. THRESHOLDS
# ============================================================

DECLINE_WINDOW = 20

RAPID_DECLINE_THRESHOLD = -1.0

SPEI1_ONSET_THRESHOLD = -1.2

SPEI3_DROUGHT_THRESHOLD = -1.0

SPEI3_TERMINATION_THRESHOLD = -0.5


# ============================================================
# 3. OPEN SPEI DATA
# ============================================================

ds = xr.open_dataset(input_file)

print("=" * 70)
print("REFINED FLASH DROUGHT EVENT ANALYSIS")
print("=" * 70)

print("\nVariables:")
print(list(ds.data_vars))

print("\nDimensions:")
print(dict(ds.sizes))

print(
    "\nFirst date:",
    ds.time.values[0]
)

print(
    "Last date:",
    ds.time.values[-1]
)


if "spei1" not in ds.data_vars:
    raise ValueError("spei1 not found.")

if "spei3" not in ds.data_vars:
    raise ValueError("spei3 not found.")


# ============================================================
# 4. COMPLETE HARVEST YEARS
# ============================================================

harvest_years = np.arange(
    1982,
    2026
)

n_harvest_years = len(
    harvest_years
)

print(
    "\nComplete harvest years:",
    harvest_years[0],
    "to",
    harvest_years[-1]
)

print(
    "Number of harvest years:",
    n_harvest_years
)


# ============================================================
# 5. OUTPUT BLOCK LISTS
# ============================================================

frequency_blocks = []

total_duration_blocks = []

max_duration_blocks = []

cumulative_intensity_blocks = []

peak_intensity_blocks = []


nlat = ds.sizes["lat"]

nlon = ds.sizes["lon"]


print(
    "\nLatitude rows:",
    nlat
)

print(
    "Block size:",
    block_size
)


# ============================================================
# 6. EVENT-TRACKING FUNCTION FOR ONE PIXEL
# ============================================================

def track_flash_drought_events(
    spei1_series,
    spei3_series,
    times
):

    """
    Identify distinct flash drought events.

    Returns arrays for harvest years 1982-2025:
      frequency
      total duration
      maximum duration
      cumulative intensity
      peak intensity
    """

    # --------------------------------------------------------
    # Initialise outputs
    # --------------------------------------------------------

    frequency = np.zeros(
        n_harvest_years,
        dtype=np.int16
    )

    total_duration = np.zeros(
        n_harvest_years,
        dtype=np.int16
    )

    max_duration = np.zeros(
        n_harvest_years,
        dtype=np.int16
    )

    cumulative_intensity = np.zeros(
        n_harvest_years,
        dtype=np.float32
    )

    # NaN because some years may have no events
    peak_intensity = np.full(
        n_harvest_years,
        np.nan,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # 20-day SPEI-1 change
    # --------------------------------------------------------

    change20 = np.full(
        len(spei1_series),
        np.nan,
        dtype=np.float32
    )

    change20[
        DECLINE_WINDOW:
    ] = (
        spei1_series[DECLINE_WINDOW:]
        -
        spei1_series[:-DECLINE_WINDOW]
    )


    # --------------------------------------------------------
    # Potential onset trigger
    # --------------------------------------------------------

    onset_trigger = (
        (change20 <= RAPID_DECLINE_THRESHOLD)
        &
        (spei1_series <= SPEI1_ONSET_THRESHOLD)
    )


    # --------------------------------------------------------
    # Growing-season mask + harvest year
    # --------------------------------------------------------

    months = np.array(
        [
            t.astype("datetime64[M]")
            .astype(int) % 12 + 1

            for t in times
        ]
    )


    years = np.array(
        [
            t.astype("datetime64[Y]")
            .astype(int) + 1970

            for t in times
        ]
    )


    growing = np.isin(
        months,
        [10, 11, 12, 1, 2, 3, 4]
    )


    harvest = (
        years
        +
        (months >= 10).astype(int)
    )


    # --------------------------------------------------------
    # Event scanning
    # --------------------------------------------------------

    i = 0

    n = len(times)


    while i < n:

        # Must be in maize season
        if not growing[i]:

            i += 1
            continue


        # Must be a valid onset trigger
        if (
            not onset_trigger[i]
            or np.isnan(spei3_series[i])
        ):

            i += 1
            continue


        # ----------------------------------------------------
        # After SPEI-1 onset trigger, look for SPEI-3 reaching
        # the core drought state <= -1.0
        # ----------------------------------------------------

        event_harvest_year = harvest[i]

        if (
            event_harvest_year < 1982
            or event_harvest_year > 2025
        ):

            i += 1
            continue


        j = i


        # Search forward within the SAME harvest season
        # until SPEI-3 reaches <= -1.0
        while (
            j < n
            and growing[j]
            and harvest[j] == event_harvest_year
            and (
                np.isnan(spei3_series[j])
                or spei3_series[j]
                > SPEI3_DROUGHT_THRESHOLD
            )
        ):

            # Do not let a trigger wait indefinitely.
            # Restrict confirmation to 20 days.
            if j - i > DECLINE_WINDOW:
                break

            j += 1


        # No SPEI-3 drought confirmation found
        if (
            j >= n
            or not growing[j]
            or harvest[j] != event_harvest_year
            or j - i > DECLINE_WINDOW
            or np.isnan(spei3_series[j])
            or spei3_series[j]
            > SPEI3_DROUGHT_THRESHOLD
        ):

            i += 1
            continue


        # ----------------------------------------------------
        # Confirmed event starts at SPEI-3 <= -1.0
        # ----------------------------------------------------

        event_start = j

        k = event_start


        # Continue until SPEI-3 > -0.5
        while k < n:

            if not growing[k]:

                break

            if harvest[k] != event_harvest_year:

                break

            if np.isnan(spei3_series[k]):

                break

            if (
                spei3_series[k]
                > SPEI3_TERMINATION_THRESHOLD
            ):

                break

            k += 1


        # ----------------------------------------------------
        # Event interval:
        # start = first SPEI-3 <= -1.0
        # end   = last day <= -0.5
        # ----------------------------------------------------

        event_end_exclusive = k

        duration = (
            event_end_exclusive
            -
            event_start
        )


        if duration <= 0:

            i += 1
            continue


        event_values = spei3_series[
            event_start:event_end_exclusive
        ]


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        idx = (
            event_harvest_year
            -
            1982
        )


        frequency[idx] += 1


        total_duration[idx] += duration


        if duration > max_duration[idx]:

            max_duration[idx] = duration


        # Cumulative magnitude:
        # integrate deficit below zero across the full
        # event interval, including recovery phase.
        cumulative_intensity[idx] += np.nansum(
            np.maximum(
                -event_values,
                0
            )
        )


        event_peak = np.nanmin(
            event_values
        )


        if np.isnan(
            peak_intensity[idx]
        ):

            peak_intensity[idx] = event_peak

        else:

            peak_intensity[idx] = min(
                peak_intensity[idx],
                event_peak
            )


        # ----------------------------------------------------
        # Move pointer to event termination so the same event
        # cannot be counted repeatedly
        # ----------------------------------------------------

        i = max(
            event_end_exclusive,
            i + 1
        )


    return (
        frequency,
        total_duration,
        max_duration,
        cumulative_intensity,
        peak_intensity
    )


# ============================================================
# 7. PROCESS LATITUDE BLOCKS
# ============================================================

for start in range(
    0,
    nlat,
    block_size
):

    end = min(
        start + block_size,
        nlat
    )


    print(
        f"\nProcessing latitude rows "
        f"{start}:{end} of {nlat}"
    )


    spei1_block = (
        ds["spei1"]
        .isel(
            lat=slice(
                start,
                end
            )
        )
        .load()
        .values
        .astype(
            np.float32
        )
    )


    spei3_block = (
        ds["spei3"]
        .isel(
            lat=slice(
                start,
                end
            )
        )
        .load()
        .values
        .astype(
            np.float32
        )
    )


    times = ds.time.values


    block_lat_count = (
        end - start
    )


    # --------------------------------------------------------
    # Allocate outputs for this block
    # --------------------------------------------------------

    frequency_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.int16
    )


    total_duration_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.int16
    )


    max_duration_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.int16
    )


    cumulative_intensity_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.float32
    )


    peak_intensity_block = np.full(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        np.nan,
        dtype=np.float32
    )


    # ========================================================
    # 8. PIXEL-BY-PIXEL EVENT TRACKING
    # ========================================================

    for ilat in range(
        block_lat_count
    ):

        for ilon in range(
            nlon
        ):


            s1 = spei1_block[
                :,
                ilat,
                ilon
            ]


            s3 = spei3_block[
                :,
                ilat,
                ilon
            ]


            # Skip pixels with no valid data
            if (
                np.all(np.isnan(s1))
                or np.all(np.isnan(s3))
            ):

                continue


            (
                freq,
                total_dur,
                max_dur,
                cum_int,
                peak_int
            ) = track_flash_drought_events(
                s1,
                s3,
                times
            )


            frequency_block[
                :,
                ilat,
                ilon
            ] = freq


            total_duration_block[
                :,
                ilat,
                ilon
            ] = total_dur


            max_duration_block[
                :,
                ilat,
                ilon
            ] = max_dur


            cumulative_intensity_block[
                :,
                ilat,
                ilon
            ] = cum_int


            peak_intensity_block[
                :,
                ilat,
                ilon
            ] = peak_int


    # ========================================================
    # 9. CONVERT BLOCKS TO XARRAY
    # ========================================================

    coords = {
        "harvest_year": harvest_years,

        "lat": ds.lat.values[
            start:end
        ],

        "lon": ds.lon.values
    }


    frequency_blocks.append(
        xr.DataArray(
            frequency_block,
            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),
            coords=coords
        )
    )


    total_duration_blocks.append(
        xr.DataArray(
            total_duration_block,
            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),
            coords=coords
        )
    )


    max_duration_blocks.append(
        xr.DataArray(
            max_duration_block,
            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),
            coords=coords
        )
    )


    cumulative_intensity_blocks.append(
        xr.DataArray(
            cumulative_intensity_block,
            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),
            coords=coords
        )
    )


    peak_intensity_blocks.append(
        xr.DataArray(
            peak_intensity_block,
            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),
            coords=coords
        )
    )


    # --------------------------------------------------------
    # Memory cleanup
    # --------------------------------------------------------

    del spei1_block
    del spei3_block

    del frequency_block
    del total_duration_block
    del max_duration_block

    del cumulative_intensity_block
    del peak_intensity_block

    gc.collect()


# ============================================================
# 10. COMBINE LATITUDE BLOCKS
# ============================================================

print(
    "\nCombining event metric blocks..."
)


fd_frequency = xr.concat(
    frequency_blocks,
    dim="lat"
)


fd_total_duration = xr.concat(
    total_duration_blocks,
    dim="lat"
)


fd_max_duration = xr.concat(
    max_duration_blocks,
    dim="lat"
)


fd_cumulative_intensity = xr.concat(
    cumulative_intensity_blocks,
    dim="lat"
)


fd_peak_intensity = xr.concat(
    peak_intensity_blocks,
    dim="lat"
)


# ============================================================
# 11. NAME VARIABLES
# ============================================================

fd_frequency.name = (
    "fd_frequency"
)

fd_total_duration.name = (
    "fd_total_duration"
)

fd_max_duration.name = (
    "fd_max_duration"
)

fd_cumulative_intensity.name = (
    "fd_cumulative_intensity"
)

fd_peak_intensity.name = (
    "fd_peak_intensity"
)


# ============================================================
# 12. ATTRIBUTES
# ============================================================

fd_frequency.attrs[
    "long_name"
] = (
    "Number of distinct flash drought events "
    "per maize harvest year"
)


fd_total_duration.attrs[
    "long_name"
] = (
    "Total number of flash drought event days "
    "per maize harvest year"
)

fd_total_duration.attrs[
    "units"
] = "days"


fd_max_duration.attrs[
    "long_name"
] = (
    "Maximum duration of a single flash drought event"
)

fd_max_duration.attrs[
    "units"
] = "days"


fd_cumulative_intensity.attrs[
    "long_name"
] = (
    "Cumulative SPEI-3 deficit magnitude "
    "during flash drought events"
)


fd_peak_intensity.attrs[
    "long_name"
] = (
    "Lowest SPEI-3 value reached "
    "during flash drought events"
)


# ============================================================
# 13. CREATE FINAL DATASET
# ============================================================

flash_drought_events = xr.Dataset(
    {
        "fd_frequency":
            fd_frequency,

        "fd_total_duration":
            fd_total_duration,

        "fd_max_duration":
            fd_max_duration,

        "fd_cumulative_intensity":
            fd_cumulative_intensity,

        "fd_peak_intensity":
            fd_peak_intensity
    }
)


# ============================================================
# 14. FINAL CHECK
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL FLASH DROUGHT EVENT METRICS"
)

print(
    "=" * 70
)


print(
    "Dimensions:"
)

print(
    dict(
        flash_drought_events.sizes
    )
)


print(
    "\nFirst harvest year:",
    int(
        flash_drought_events
        .harvest_year
        .values[0]
    )
)


print(
    "Last harvest year:",
    int(
        flash_drought_events
        .harvest_year
        .values[-1]
    )
)


print(
    "\nMaximum event frequency:",
    int(
        fd_frequency.max().values
    )
)


print(
    "Maximum total duration:",
    int(
        fd_total_duration.max().values
    )
)


print(
    "Maximum single-event duration:",
    int(
        fd_max_duration.max().values
    )
)


print(
    "Maximum cumulative intensity:",
    float(
        fd_cumulative_intensity
        .max()
        .values
    )
)


print(
    "Lowest peak SPEI-3:",
    float(
        fd_peak_intensity
        .min(
            skipna=True
        )
        .values
    )
)


# ============================================================
# 15. TEST LOCATION
# ============================================================

test = flash_drought_events.sel(
    lat=-27.95,
    lon=21.05,
    method="nearest"
)


print(
    "\nTest location"
)

print(
    "-" * 40
)


print(
    "Latitude:",
    float(test.lat)
)


print(
    "Longitude:",
    float(test.lon)
)


print(
    "\nFirst 10 harvest years:"
)

print(
    test.harvest_year.values[:10]
)


print(
    "\nFrequency:"
)

print(
    test.fd_frequency.values[:10]
)


print(
    "\nTotal duration:"
)

print(
    test.fd_total_duration.values[:10]
)


print(
    "\nMaximum event duration:"
)

print(
    test.fd_max_duration.values[:10]
)


print(
    "\nCumulative intensity:"
)

print(
    test.fd_cumulative_intensity.values[:10]
)


print(
    "\nPeak intensity:"
)

print(
    test.fd_peak_intensity.values[:10]
)


# ============================================================
# 16. SAVE
# ============================================================

print(
    "\nSaving refined flash drought events..."
)


flash_drought_events.to_netcdf(
    output_file
)


print(
    "\nSaved successfully:"
)

print(
    output_file
)


print(
    "\nDONE!"
)

REFINED FLASH DROUGHT EVENT ANALYSIS

Variables:
['spei1', 'spei3']

Dimensions:
{'time': 16436, 'lat': 141, 'lon': 220}

First date: 1981-01-01T00:00:00.000000000
Last date: 2025-12-31T00:00:00.000000000

Complete harvest years: 1982 to 2025
Number of harvest years: 44

Latitude rows: 141
Block size: 3

Processing latitude rows 0:3 of 141

Processing latitude rows 3:6 of 141

Processing latitude rows 6:9 of 141

Processing latitude rows 9:12 of 141

Processing latitude rows 12:15 of 141

Processing latitude rows 15:18 of 141

Processing latitude rows 18:21 of 141

Processing latitude rows 21:24 of 141

Processing latitude rows 24:27 of 141

Processing latitude rows 27:30 of 141

Processing latitude rows 30:33 of 141

Processing latitude rows 33:36 of 141

Processing latitude rows 36:39 of 141

Processing latitude rows 39:42 of 141

Processing latitude rows 42:45 of 141

Processing latitude rows 45:48 of 141

Processing latitude rows 48:51 of 141

Processing latitude rows 51:54 of 141


In [15]:
# ============================================================
# FIND THE LONGEST FLASH DROUGHT EVENT
# ============================================================

fd = xr.open_dataset(
    r"C:\Users\franc\flash_drought_events_1982_2025.nc"
)

max_duration = fd["fd_max_duration"].max()

print("Maximum duration:", int(max_duration.values))

# Find location(s) where maximum occurs
where_max = fd["fd_max_duration"].where(
    fd["fd_max_duration"] == max_duration,
    drop=True
)

print("\nLocations/years with maximum duration:")
print(where_max)

Maximum duration: 18403200000000000

Locations/years with maximum duration:
<xarray.DataArray 'fd_max_duration' (harvest_year: 1, lat: 6, lon: 6)> Size: 288B
array([[[            'NaT',             'NaT',             'NaT',
                     'NaT',             'NaT', 18403200000000000],
        [            'NaT', 18403200000000000, 18403200000000000,
         18403200000000000, 18403200000000000,             'NaT'],
        [            'NaT', 18403200000000000, 18403200000000000,
         18403200000000000,             'NaT',             'NaT'],
        [18403200000000000,             'NaT', 18403200000000000,
         18403200000000000,             'NaT',             'NaT'],
        [18403200000000000, 18403200000000000, 18403200000000000,
         18403200000000000,             'NaT',             'NaT'],
        [            'NaT',             'NaT', 18403200000000000,
                     'NaT',             'NaT',             'NaT']]],
      dtype='timedelta64[ns]')
Coordinates

C:\Users\franc\AppData\Local\Temp\ipykernel_27560\2430880826.py:5: FutureWarning: In a future version, xarray will not decode the variable 'fd_total_duration' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  fd = xr.open_dataset(
C:\Users\franc\AppData\Local\Temp\ipykernel_27560\2430880826.py:5: FutureWarning: In a future version, xarray will not decode the variable 'fd_max_duration' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now

In [16]:
# Stack spatial/year dimensions to locate maximum
stacked = fd["fd_max_duration"].stack(
    point=("harvest_year", "lat", "lon")
)

idx = int(stacked.argmax().values)

longest = stacked.isel(point=idx)

print("\nLongest event")
print("-------------------------")

print(
    "Harvest year:",
    int(longest.harvest_year.values)
)

print(
    "Latitude:",
    float(longest.lat.values)
)

print(
    "Longitude:",
    float(longest.lon.values)
)

print(
    "Duration:",
    int(longest.values),
    "days"
)


Longest event
-------------------------
Harvest year: 2020
Latitude: -28.150000000003516
Longitude: 20.149999999988637
Duration: 18403200000000000 days


In [17]:
import xarray as xr

fd = xr.open_dataset(
    r"C:\Users\franc\flash_drought_events_1982_2025.nc",
    decode_timedelta=False
)

print(fd["fd_max_duration"].dtype)
print(
    "Maximum duration:",
    int(fd["fd_max_duration"].max().values),
    "days"
)

int16
Maximum duration: 213 days


In [18]:
stacked = fd["fd_max_duration"].stack(
    point=("harvest_year", "lat", "lon")
)

idx = int(stacked.argmax().values)

longest = stacked.isel(point=idx)

print("\nLongest event")
print("-------------------------")

print(
    "Harvest year:",
    int(longest.harvest_year.values)
)

print(
    "Latitude:",
    float(longest.lat.values)
)

print(
    "Longitude:",
    float(longest.lon.values)
)

print(
    "Duration:",
    int(longest.values),
    "days"
)


Longest event
-------------------------
Harvest year: 2020
Latitude: -28.150000000003516
Longitude: 20.149999999988637
Duration: 213 days


In [19]:
# Remove the CF-style "days" units that trigger timedelta decoding

fd["fd_total_duration"].attrs.pop("units", None)
fd["fd_max_duration"].attrs.pop("units", None)

fd["fd_total_duration"].attrs[
    "description"
] = "Total number of flash drought event days per harvest year"

fd["fd_max_duration"].attrs[
    "description"
] = "Maximum number of days in a single flash drought event"

fixed_file = (
    r"C:\Users\franc\flash_drought_events_1982_2025_fixed.nc"
)

fd.to_netcdf(fixed_file)

print("Saved corrected file:")
print(fixed_file)

Saved corrected file:
C:\Users\franc\flash_drought_events_1982_2025_fixed.nc


In [20]:
# ============================================================
# INSPECT THE 2020 LONGEST FLASH DROUGHT EVENT
# ============================================================

import xarray as xr
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. OPEN SPEI DATA
# ------------------------------------------------------------

spei_file = r"C:\Users\franc\spei1_spei3_1981_2025.nc"

ds = xr.open_dataset(spei_file)

# Longest-event location
lat_event = -28.15
lon_event = 20.15

# ------------------------------------------------------------
# 2. EXTRACT PIXEL
# ------------------------------------------------------------

s1 = ds["spei1"].sel(
    lat=lat_event,
    lon=lon_event,
    method="nearest"
)

s3 = ds["spei3"].sel(
    lat=lat_event,
    lon=lon_event,
    method="nearest"
)

print("Actual pixel:")
print("Latitude:", float(s1.lat))
print("Longitude:", float(s1.lon))

# ------------------------------------------------------------
# 3. TAKE THE 2019–2020 MAIZE SEASON
# ------------------------------------------------------------

season_start = "2019-10-01"
season_end = "2020-04-30"

s1_season = s1.sel(
    time=slice(season_start, season_end)
)

s3_season = s3.sel(
    time=slice(season_start, season_end)
)

print("\nSeason:")
print("First date:", s3_season.time.values[0])
print("Last date :", s3_season.time.values[-1])
print("Number of days:", s3_season.sizes["time"])

# ------------------------------------------------------------
# 4. CALCULATE 20-DAY SPEI-1 CHANGE
# ------------------------------------------------------------

change20 = (
    s1
    -
    s1.shift(time=20)
)

change20_season = change20.sel(
    time=slice(season_start, season_end)
)

# ------------------------------------------------------------
# 5. FLASH-DROUGHT ONSET TRIGGER
# ------------------------------------------------------------

onset = (
    (change20_season <= -1.0)
    &
    (s1_season <= -1.2)
)

onset_dates = s1_season.time.where(
    onset,
    drop=True
)

print("\nPotential SPEI-1 onset dates:")
print(onset_dates.values)

# ------------------------------------------------------------
# 6. SPEI-3 CORE DROUGHT DATES
# ------------------------------------------------------------

core_drought = (
    s3_season <= -1.0
)

core_dates = s3_season.time.where(
    core_drought,
    drop=True
)

print("\nFirst SPEI-3 <= -1.0 date:")

if core_dates.size > 0:
    print(core_dates.values[0])
else:
    print("None")

# ------------------------------------------------------------
# 7. FIND FIRST CORE-DROUGHT DATE AFTER AN ONSET TRIGGER
# ------------------------------------------------------------

confirmed_start = None
trigger_date = None

for onset_date in onset_dates.values:

    search_end = (
        pd.Timestamp(onset_date)
        + pd.Timedelta(days=20)
    )

    candidate = s3.sel(
        time=slice(
            str(pd.Timestamp(onset_date).date()),
            str(search_end.date())
        )
    )

    candidate_core = candidate.where(
        candidate <= -1.0,
        drop=True
    )

    if candidate_core.time.size > 0:

        trigger_date = onset_date
        confirmed_start = candidate_core.time.values[0]
        break

print("\nConfirmed flash-drought trigger:")
print(trigger_date)

print("Confirmed SPEI-3 event start:")
print(confirmed_start)

# ------------------------------------------------------------
# 8. FIND TERMINATION (> -0.5)
# ------------------------------------------------------------

if confirmed_start is not None:

    after_start = s3.sel(
        time=slice(
            str(pd.Timestamp(confirmed_start).date()),
            "2020-12-31"
        )
    )

    recovery = after_start.where(
        after_start > -0.5,
        drop=True
    )

    if recovery.time.size > 0:

        termination_date = recovery.time.values[0]

        print("\nFirst SPEI-3 > -0.5 after event start:")
        print(termination_date)

        true_duration = (
            pd.Timestamp(termination_date)
            -
            pd.Timestamp(confirmed_start)
        ).days

        print(
            "Duration until termination:",
            true_duration,
            "days"
        )

    else:

        termination_date = None

        print(
            "\nSPEI-3 never rises above -0.5 "
            "before the end of available data."
        )

# ------------------------------------------------------------
# 9. CHECK VALUE ON APRIL 30
# ------------------------------------------------------------

apr30 = s3.sel(
    time="2020-04-30"
)

print("\nSPEI-3 on 2020-04-30:")
print(float(apr30.values))

# ------------------------------------------------------------
# 10. PRINT LAST 15 DAYS OF THE GROWING SEASON
# ------------------------------------------------------------

print("\nLast 15 days of 2019–2020 growing season:")
print("-----------------------------------------")

for t, value in zip(
    s3_season.time.values[-15:],
    s3_season.values[-15:]
):

    print(
        str(t)[:10],
        round(float(value), 3)
    )

Actual pixel:
Latitude: -28.150000000003516
Longitude: 20.149999999988637

Season:
First date: 2019-10-01T00:00:00.000000000
Last date : 2020-04-30T00:00:00.000000000
Number of days: 213

Potential SPEI-1 onset dates:
['2019-10-01T00:00:00.000000000' '2019-10-02T00:00:00.000000000'
 '2019-10-03T00:00:00.000000000' '2019-10-04T00:00:00.000000000'
 '2019-10-05T00:00:00.000000000' '2019-10-06T00:00:00.000000000'
 '2019-10-07T00:00:00.000000000' '2019-10-08T00:00:00.000000000'
 '2019-10-09T00:00:00.000000000' '2019-10-11T00:00:00.000000000'
 '2019-10-12T00:00:00.000000000' '2019-10-13T00:00:00.000000000'
 '2019-10-14T00:00:00.000000000' '2019-10-15T00:00:00.000000000'
 '2019-10-16T00:00:00.000000000' '2019-12-04T00:00:00.000000000'
 '2020-02-09T00:00:00.000000000' '2020-02-10T00:00:00.000000000'
 '2020-03-30T00:00:00.000000000' '2020-03-31T00:00:00.000000000'
 '2020-04-01T00:00:00.000000000' '2020-04-02T00:00:00.000000000'
 '2020-04-03T00:00:00.000000000' '2020-04-04T00:00:00.000000000'
 '

In [21]:
# ============================================================
# FINAL REFINED FLASH DROUGHT EVENT TRACKING
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

# ============================================================
# 1. PATHS
# ============================================================

input_file = r"C:\Users\franc\spei1_spei3_1981_2025.nc"

output_file = (
    r"C:\Users\franc\flash_drought_events_1982_2025_final.nc"
)

block_size = 3


# ============================================================
# 2. THRESHOLDS
# ============================================================

DECLINE_WINDOW = 20

RAPID_DECLINE_THRESHOLD = -1.0

SPEI1_ONSET_THRESHOLD = -1.2

SPEI3_DROUGHT_THRESHOLD = -1.0

SPEI3_TERMINATION_THRESHOLD = -0.5


# ============================================================
# 3. OPEN DATA
# ============================================================

ds = xr.open_dataset(input_file)

print("=" * 70)
print("FINAL REFINED FLASH DROUGHT EVENT ANALYSIS")
print("=" * 70)

print("\nVariables:")
print(list(ds.data_vars))

print("\nDimensions:")
print(dict(ds.sizes))

print("\nFirst date:", ds.time.values[0])
print("Last date :", ds.time.values[-1])


# ============================================================
# 4. HARVEST YEARS
# ============================================================

harvest_years = np.arange(1982, 2026)

n_harvest_years = len(harvest_years)

nlat = ds.sizes["lat"]
nlon = ds.sizes["lon"]

print("\nHarvest years:", harvest_years[0], "to", harvest_years[-1])
print("Number of harvest years:", n_harvest_years)

print("\nLatitude rows:", nlat)
print("Longitude columns:", nlon)
print("Block size:", block_size)


# ============================================================
# 5. OUTPUT BLOCK LISTS
# ============================================================

frequency_blocks = []

total_duration_blocks = []

max_duration_blocks = []

cumulative_intensity_blocks = []

peak_intensity_blocks = []


# ============================================================
# 6. EVENT TRACKING FUNCTION
# ============================================================

def track_flash_drought_events(
    spei1_series,
    spei3_series,
    times
):

    # --------------------------------------------------------
    # Output arrays
    # --------------------------------------------------------

    frequency = np.zeros(
        n_harvest_years,
        dtype=np.int16
    )

    total_duration = np.zeros(
        n_harvest_years,
        dtype=np.int16
    )

    max_duration = np.zeros(
        n_harvest_years,
        dtype=np.int16
    )

    cumulative_intensity = np.zeros(
        n_harvest_years,
        dtype=np.float32
    )

    peak_intensity = np.full(
        n_harvest_years,
        np.nan,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # 20-day SPEI-1 change
    # --------------------------------------------------------

    change20 = np.full(
        len(spei1_series),
        np.nan,
        dtype=np.float32
    )

    change20[DECLINE_WINDOW:] = (
        spei1_series[DECLINE_WINDOW:]
        -
        spei1_series[:-DECLINE_WINDOW]
    )


    # --------------------------------------------------------
    # Rapid flash-drought onset trigger
    # --------------------------------------------------------

    onset_trigger = (
        (change20 <= RAPID_DECLINE_THRESHOLD)
        &
        (spei1_series <= SPEI1_ONSET_THRESHOLD)
    )


    # --------------------------------------------------------
    # Calendar information
    # --------------------------------------------------------

    months = np.array(
        [
            t.astype("datetime64[M]").astype(int) % 12 + 1
            for t in times
        ]
    )

    years = np.array(
        [
            t.astype("datetime64[Y]").astype(int) + 1970
            for t in times
        ]
    )


    # --------------------------------------------------------
    # Maize growing season
    # --------------------------------------------------------

    growing = np.isin(
        months,
        [10, 11, 12, 1, 2, 3, 4]
    )


    # --------------------------------------------------------
    # Harvest year
    #
    # Oct-Dec 2019 -> 2020
    # Jan-Apr 2020 -> 2020
    # --------------------------------------------------------

    harvest = (
        years
        +
        (months >= 10).astype(int)
    )


    # ========================================================
    # EVENT SCANNING
    # ========================================================

    i = 0

    n = len(times)


    while i < n:

        # ----------------------------------------------------
        # Onset must occur during maize growing season
        # ----------------------------------------------------

        if not growing[i]:
            i += 1
            continue


        # ----------------------------------------------------
        # Must meet rapid SPEI-1 onset trigger
        # ----------------------------------------------------

        if (
            not onset_trigger[i]
            or np.isnan(spei3_series[i])
        ):
            i += 1
            continue


        event_harvest_year = harvest[i]


        # ----------------------------------------------------
        # Keep only complete harvest years
        # ----------------------------------------------------

        if (
            event_harvest_year < 1982
            or event_harvest_year > 2025
        ):
            i += 1
            continue


        # ====================================================
        # SEARCH FOR SPEI-3 <= -1.0 CONFIRMATION
        # ====================================================

        j = i


        while (
            j < n
            and growing[j]
            and harvest[j] == event_harvest_year
        ):

            if np.isnan(spei3_series[j]):
                break

            if (
                spei3_series[j]
                <= SPEI3_DROUGHT_THRESHOLD
            ):
                break

            # Confirmation must occur within 20 days
            if (j - i) >= DECLINE_WINDOW:
                break

            j += 1


        # ----------------------------------------------------
        # No valid SPEI-3 drought confirmation
        # ----------------------------------------------------

        if (
            j >= n
            or np.isnan(spei3_series[j])
            or not growing[j]
            or harvest[j] != event_harvest_year
            or spei3_series[j] > SPEI3_DROUGHT_THRESHOLD
            or (j - i) > DECLINE_WINDOW
        ):

            i += 1
            continue


        # ====================================================
        # CONFIRMED FLASH DROUGHT EVENT
        # ====================================================

        event_start = j


        # ====================================================
        # TRACK EVENT UNTIL SPEI-3 > -0.5
        #
        # IMPORTANT:
        # We DO NOT stop at the end of April.
        #
        # Once an event begins during the maize season,
        # tracking continues until actual recovery.
        # ====================================================

        k = event_start


        while k < n:

            if np.isnan(spei3_series[k]):
                break

            # Supervisor termination criterion
            if (
                spei3_series[k]
                > SPEI3_TERMINATION_THRESHOLD
            ):
                break

            k += 1


        # ----------------------------------------------------
        # k = first day ABOVE -0.5
        #
        # Event days are:
        # event_start ... k-1
        # ----------------------------------------------------

        event_end_exclusive = k


        duration = (
            event_end_exclusive
            -
            event_start
        )


        if duration <= 0:

            i += 1
            continue


        event_values = spei3_series[
            event_start:event_end_exclusive
        ]


        # ====================================================
        # METRICS
        # ====================================================

        idx = (
            event_harvest_year
            -
            1982
        )


        # ----------------------------------------------------
        # Frequency:
        # number of distinct events
        # ----------------------------------------------------

        frequency[idx] += 1


        # ----------------------------------------------------
        # Total duration
        # ----------------------------------------------------

        total_duration[idx] += duration


        # ----------------------------------------------------
        # Longest individual event
        # ----------------------------------------------------

        if duration > max_duration[idx]:

            max_duration[idx] = duration


        # ----------------------------------------------------
        # Cumulative intensity
        #
        # Entire event interval is used,
        # including recovery phase <= -0.5.
        # ----------------------------------------------------

        cumulative_intensity[idx] += np.nansum(
            np.maximum(
                -event_values,
                0
            )
        )


        # ----------------------------------------------------
        # Peak intensity:
        # lowest SPEI-3 reached during event
        # ----------------------------------------------------

        event_peak = np.nanmin(
            event_values
        )


        if np.isnan(
            peak_intensity[idx]
        ):

            peak_intensity[idx] = event_peak

        else:

            peak_intensity[idx] = min(
                peak_intensity[idx],
                event_peak
            )


        # ----------------------------------------------------
        # Move to first day after event termination marker
        # Prevents duplicate counting
        # ----------------------------------------------------

        i = max(
            event_end_exclusive + 1,
            i + 1
        )


    return (
        frequency,
        total_duration,
        max_duration,
        cumulative_intensity,
        peak_intensity
    )


# ============================================================
# 7. PROCESS LATITUDE BLOCKS
# ============================================================

for start in range(
    0,
    nlat,
    block_size
):

    end = min(
        start + block_size,
        nlat
    )


    print(
        f"\nProcessing latitude rows "
        f"{start}:{end} of {nlat}"
    )


    spei1_block = (
        ds["spei1"]
        .isel(
            lat=slice(start, end)
        )
        .load()
        .values
        .astype(np.float32)
    )


    spei3_block = (
        ds["spei3"]
        .isel(
            lat=slice(start, end)
        )
        .load()
        .values
        .astype(np.float32)
    )


    times = ds.time.values


    block_lat_count = (
        end - start
    )


    # --------------------------------------------------------
    # Allocate block outputs
    # --------------------------------------------------------

    frequency_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.int16
    )


    total_duration_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.int16
    )


    max_duration_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.int16
    )


    cumulative_intensity_block = np.zeros(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        dtype=np.float32
    )


    peak_intensity_block = np.full(
        (
            n_harvest_years,
            block_lat_count,
            nlon
        ),
        np.nan,
        dtype=np.float32
    )


    # ========================================================
    # 8. PIXEL-BY-PIXEL TRACKING
    # ========================================================

    for ilat in range(
        block_lat_count
    ):

        for ilon in range(
            nlon
        ):


            s1 = spei1_block[
                :,
                ilat,
                ilon
            ]


            s3 = spei3_block[
                :,
                ilat,
                ilon
            ]


            # Skip empty pixels
            if (
                np.all(np.isnan(s1))
                or
                np.all(np.isnan(s3))
            ):

                continue


            (
                freq,
                total_dur,
                max_dur,
                cum_int,
                peak_int
            ) = track_flash_drought_events(
                s1,
                s3,
                times
            )


            frequency_block[
                :,
                ilat,
                ilon
            ] = freq


            total_duration_block[
                :,
                ilat,
                ilon
            ] = total_dur


            max_duration_block[
                :,
                ilat,
                ilon
            ] = max_dur


            cumulative_intensity_block[
                :,
                ilat,
                ilon
            ] = cum_int


            peak_intensity_block[
                :,
                ilat,
                ilon
            ] = peak_int


    # ========================================================
    # 9. CONVERT BLOCKS TO XARRAY
    # ========================================================

    coords = {

        "harvest_year":
            harvest_years,

        "lat":
            ds.lat.values[
                start:end
            ],

        "lon":
            ds.lon.values
    }


    frequency_blocks.append(
        xr.DataArray(
            frequency_block,

            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),

            coords=coords
        )
    )


    total_duration_blocks.append(
        xr.DataArray(
            total_duration_block,

            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),

            coords=coords
        )
    )


    max_duration_blocks.append(
        xr.DataArray(
            max_duration_block,

            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),

            coords=coords
        )
    )


    cumulative_intensity_blocks.append(
        xr.DataArray(
            cumulative_intensity_block,

            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),

            coords=coords
        )
    )


    peak_intensity_blocks.append(
        xr.DataArray(
            peak_intensity_block,

            dims=(
                "harvest_year",
                "lat",
                "lon"
            ),

            coords=coords
        )
    )


    # --------------------------------------------------------
    # Clean memory
    # --------------------------------------------------------

    del spei1_block
    del spei3_block

    del frequency_block
    del total_duration_block
    del max_duration_block

    del cumulative_intensity_block
    del peak_intensity_block

    gc.collect()


# ============================================================
# 10. COMBINE BLOCKS
# ============================================================

print(
    "\nCombining event metric blocks..."
)


fd_frequency = xr.concat(
    frequency_blocks,
    dim="lat"
)


fd_total_duration = xr.concat(
    total_duration_blocks,
    dim="lat"
)


fd_max_duration = xr.concat(
    max_duration_blocks,
    dim="lat"
)


fd_cumulative_intensity = xr.concat(
    cumulative_intensity_blocks,
    dim="lat"
)


fd_peak_intensity = xr.concat(
    peak_intensity_blocks,
    dim="lat"
)


# ============================================================
# 11. VARIABLE NAMES
# ============================================================

fd_frequency.name = "fd_frequency"

fd_total_duration.name = "fd_total_duration"

fd_max_duration.name = "fd_max_duration"

fd_cumulative_intensity.name = (
    "fd_cumulative_intensity"
)

fd_peak_intensity.name = (
    "fd_peak_intensity"
)


# ============================================================
# 12. ATTRIBUTES
# ============================================================

fd_frequency.attrs["long_name"] = (
    "Number of distinct flash drought events "
    "per maize harvest year"
)


# IMPORTANT:
# Do not use units='days' because xarray may decode
# these integer counts as timedelta64 when reopened.

fd_total_duration.attrs["long_name"] = (
    "Total number of flash drought event days "
    "per maize harvest year"
)

fd_total_duration.attrs["description"] = (
    "Integer count of event days"
)


fd_max_duration.attrs["long_name"] = (
    "Maximum duration of a single flash drought event"
)

fd_max_duration.attrs["description"] = (
    "Integer count of event days"
)


fd_cumulative_intensity.attrs["long_name"] = (
    "Cumulative SPEI-3 deficit magnitude "
    "during complete flash drought events"
)


fd_peak_intensity.attrs["long_name"] = (
    "Lowest SPEI-3 value reached "
    "during flash drought events"
)


# ============================================================
# 13. CREATE FINAL DATASET
# ============================================================

flash_drought_events = xr.Dataset(
    {

        "fd_frequency":
            fd_frequency,

        "fd_total_duration":
            fd_total_duration,

        "fd_max_duration":
            fd_max_duration,

        "fd_cumulative_intensity":
            fd_cumulative_intensity,

        "fd_peak_intensity":
            fd_peak_intensity

    }
)


# ============================================================
# 14. FINAL CHECK
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL FLASH DROUGHT EVENT METRICS"
)

print(
    "=" * 70
)


print(
    "Dimensions:"
)

print(
    dict(
        flash_drought_events.sizes
    )
)


print(
    "\nFirst harvest year:",
    int(
        flash_drought_events
        .harvest_year
        .values[0]
    )
)


print(
    "Last harvest year:",
    int(
        flash_drought_events
        .harvest_year
        .values[-1]
    )
)


print(
    "\nMaximum event frequency:",
    int(
        fd_frequency.max().values
    )
)


print(
    "Maximum total duration:",
    int(
        fd_total_duration.max().values
    )
)


print(
    "Maximum single-event duration:",
    int(
        fd_max_duration.max().values
    )
)


print(
    "Maximum cumulative intensity:",
    float(
        fd_cumulative_intensity
        .max()
        .values
    )
)


print(
    "Lowest peak SPEI-3:",
    float(
        fd_peak_intensity
        .min(
            skipna=True
        )
        .values
    )
)


# ============================================================
# 15. TEST LOCATION
# ============================================================

test = flash_drought_events.sel(

    lat=-27.95,

    lon=21.05,

    method="nearest"
)


print(
    "\nTest location"
)

print(
    "-" * 40
)


print(
    "Latitude:",
    float(test.lat)
)


print(
    "Longitude:",
    float(test.lon)
)


print(
    "\nFirst 10 harvest years:"
)

print(
    test.harvest_year.values[:10]
)


print(
    "\nFrequency:"
)

print(
    test.fd_frequency.values[:10]
)


print(
    "\nTotal duration:"
)

print(
    test.fd_total_duration.values[:10]
)


print(
    "\nMaximum event duration:"
)

print(
    test.fd_max_duration.values[:10]
)


print(
    "\nCumulative intensity:"
)

print(
    test.fd_cumulative_intensity.values[:10]
)


print(
    "\nPeak intensity:"
)

print(
    test.fd_peak_intensity.values[:10]
)


# ============================================================
# 16. SAVE FINAL DATASET
# ============================================================

print(
    "\nSaving FINAL flash drought events..."
)


flash_drought_events.to_netcdf(
    output_file
)


print(
    "\nSaved successfully:"
)

print(
    output_file
)


print(
    "\nDONE!"
)

FINAL REFINED FLASH DROUGHT EVENT ANALYSIS

Variables:
['spei1', 'spei3']

Dimensions:
{'time': 16436, 'lat': 141, 'lon': 220}

First date: 1981-01-01T00:00:00.000000000
Last date : 2025-12-31T00:00:00.000000000

Harvest years: 1982 to 2025
Number of harvest years: 44

Latitude rows: 141
Longitude columns: 220
Block size: 3

Processing latitude rows 0:3 of 141

Processing latitude rows 3:6 of 141

Processing latitude rows 6:9 of 141

Processing latitude rows 9:12 of 141

Processing latitude rows 12:15 of 141

Processing latitude rows 15:18 of 141

Processing latitude rows 18:21 of 141

Processing latitude rows 21:24 of 141

Processing latitude rows 24:27 of 141

Processing latitude rows 27:30 of 141

Processing latitude rows 30:33 of 141

Processing latitude rows 33:36 of 141

Processing latitude rows 36:39 of 141

Processing latitude rows 39:42 of 141

Processing latitude rows 42:45 of 141

Processing latitude rows 45:48 of 141

Processing latitude rows 48:51 of 141

Processing latitu

In [22]:
# ============================================================
# FIND THE LONGEST FINAL FLASH-DROUGHT EVENT
# ============================================================

import xarray as xr

fd = xr.open_dataset(
    r"C:\Users\franc\flash_drought_events_1982_2025_final.nc",
    decode_timedelta=False
)

stacked = fd["fd_max_duration"].stack(
    point=("harvest_year", "lat", "lon")
)

idx = int(stacked.argmax().values)

longest = stacked.isel(point=idx)

print("Longest event")
print("-------------------------")

print(
    "Harvest year:",
    int(longest.harvest_year.values)
)

print(
    "Latitude:",
    float(longest.lat.values)
)

print(
    "Longitude:",
    float(longest.lon.values)
)

print(
    "Duration:",
    int(longest.values),
    "days"
)

Longest event
-------------------------
Harvest year: 2019
Latitude: -21.75000000000388
Longitude: 15.149999999988921
Duration: 732 days


In [23]:
# ============================================================
# INSPECT LONGEST EVENT: HARVEST YEAR 2019
# ============================================================

import xarray as xr
import pandas as pd
import numpy as np

spei_file = r"C:\Users\franc\spei1_spei3_1981_2025.nc"

ds = xr.open_dataset(spei_file)

lat_event = -21.75
lon_event = 15.15

s1 = ds["spei1"].sel(
    lat=lat_event,
    lon=lon_event,
    method="nearest"
)

s3 = ds["spei3"].sel(
    lat=lat_event,
    lon=lon_event,
    method="nearest"
)

print("Actual pixel:")
print("Latitude:", float(s1.lat))
print("Longitude:", float(s1.lon))

# ------------------------------------------------------------
# 20-day SPEI-1 change
# ------------------------------------------------------------

change20 = s1 - s1.shift(time=20)

onset = (
    (change20 <= -1.0)
    &
    (s1 <= -1.2)
)

# Harvest year 2019 onset season:
# Oct-Dec 2018 + Jan-Apr 2019

season_mask = (
    (
        (s1.time.dt.year == 2018)
        &
        (s1.time.dt.month >= 10)
    )
    |
    (
        (s1.time.dt.year == 2019)
        &
        (s1.time.dt.month <= 4)
    )
)

onset_dates = s1.time.where(
    onset & season_mask,
    drop=True
)

print("\nPotential onset dates:")
print(onset_dates.values)

# ------------------------------------------------------------
# Find the first valid onset followed by SPEI-3 <= -1.0
# within 20 days
# ------------------------------------------------------------

trigger_date = None
event_start = None

for onset_date in onset_dates.values:

    end_search = (
        pd.Timestamp(onset_date)
        + pd.Timedelta(days=20)
    )

    candidate = s3.sel(
        time=slice(
            str(pd.Timestamp(onset_date).date()),
            str(end_search.date())
        )
    )

    candidate_core = candidate.where(
        candidate <= -1.0,
        drop=True
    )

    if candidate_core.time.size > 0:

        trigger_date = onset_date
        event_start = candidate_core.time.values[0]
        break

print("\nTrigger date:")
print(trigger_date)

print("SPEI-3 event start:")
print(event_start)

# ------------------------------------------------------------
# Find actual termination: first SPEI-3 > -0.5
# ------------------------------------------------------------

if event_start is not None:

    after_start = s3.sel(
        time=slice(
            str(pd.Timestamp(event_start).date()),
            "2025-12-31"
        )
    )

    termination = after_start.where(
        after_start > -0.5,
        drop=True
    )

    if termination.time.size > 0:

        termination_date = termination.time.values[0]

        print("\nFirst SPEI-3 > -0.5:")
        print(termination_date)

        duration = (
            pd.Timestamp(termination_date)
            -
            pd.Timestamp(event_start)
        ).days

        print(
            "Calendar difference:",
            duration,
            "days"
        )

    else:

        termination_date = None

        print("\nNo termination found.")

# ------------------------------------------------------------
# Check whether it crosses harvest seasons
# ------------------------------------------------------------

if event_start is not None and termination_date is not None:

    event_dates = pd.date_range(
        start=pd.Timestamp(event_start),
        end=pd.Timestamp(termination_date),
        freq="D"
    )

    harvest_years_crossed = sorted(
        set(
            [
                d.year + 1 if d.month >= 10 else d.year
                for d in event_dates
            ]
        )
    )

    print("\nHarvest years crossed:")
    print(harvest_years_crossed)

# ------------------------------------------------------------
# Show SPEI-3 around start and end
# ------------------------------------------------------------

if event_start is not None:

    print("\nSPEI-3 around event start:")
    print(
        s3.sel(
            time=slice(
                str(
                    (
                        pd.Timestamp(event_start)
                        - pd.Timedelta(days=5)
                    ).date()
                ),
                str(
                    (
                        pd.Timestamp(event_start)
                        + pd.Timedelta(days=10)
                    ).date()
                )
            )
        ).values
    )

if termination_date is not None:

    print("\nSPEI-3 around termination:")
    print(
        s3.sel(
            time=slice(
                str(
                    (
                        pd.Timestamp(termination_date)
                        - pd.Timedelta(days=10)
                    ).date()
                ),
                str(
                    (
                        pd.Timestamp(termination_date)
                        + pd.Timedelta(days=5)
                    ).date()
                )
            )
        ).values
    )

Actual pixel:
Latitude: -21.75000000000388
Longitude: 15.149999999988921

Potential onset dates:
['2018-10-04T00:00:00.000000000' '2018-10-05T00:00:00.000000000'
 '2018-10-06T00:00:00.000000000' '2018-10-07T00:00:00.000000000'
 '2018-10-08T00:00:00.000000000' '2018-10-09T00:00:00.000000000'
 '2018-10-10T00:00:00.000000000' '2018-10-11T00:00:00.000000000'
 '2018-10-12T00:00:00.000000000' '2018-10-13T00:00:00.000000000'
 '2018-10-16T00:00:00.000000000' '2018-10-17T00:00:00.000000000'
 '2018-10-18T00:00:00.000000000' '2018-10-19T00:00:00.000000000'
 '2018-11-20T00:00:00.000000000' '2018-11-21T00:00:00.000000000'
 '2018-11-22T00:00:00.000000000' '2018-11-23T00:00:00.000000000'
 '2018-11-24T00:00:00.000000000' '2018-11-25T00:00:00.000000000'
 '2018-11-26T00:00:00.000000000' '2018-11-27T00:00:00.000000000'
 '2018-11-28T00:00:00.000000000' '2018-11-29T00:00:00.000000000'
 '2018-11-30T00:00:00.000000000' '2018-12-01T00:00:00.000000000'
 '2018-12-02T00:00:00.000000000' '2018-12-03T00:00:00.0000

In [24]:
# ============================================================
# CREATE DAILY FLASH-DROUGHT ACTIVE MASK
# 1 = flash drought event active, 0 = not active
# ============================================================

import os
import gc
import numpy as np
import xarray as xr

# ============================================================
# 1. INPUT / OUTPUT
# ============================================================

input_file = r"C:\Users\franc\spei1_spei3_1981_2025.nc"

output_file = (
    r"C:\Users\franc\flash_drought_daily_mask_1981_2025.nc"
)

block_size = 3


# ============================================================
# 2. THRESHOLDS
# ============================================================

DECLINE_WINDOW = 20

RAPID_DECLINE_THRESHOLD = -1.0

SPEI1_ONSET_THRESHOLD = -1.2

SPEI3_DROUGHT_THRESHOLD = -1.0

SPEI3_TERMINATION_THRESHOLD = -0.5


# ============================================================
# 3. OPEN DATA
# ============================================================

ds = xr.open_dataset(input_file)

print("=" * 70)
print("DAILY FLASH DROUGHT ACTIVE MASK")
print("=" * 70)

print("\nDimensions:")
print(dict(ds.sizes))

print("\nFirst date:", ds.time.values[0])
print("Last date :", ds.time.values[-1])


# ============================================================
# 4. CALENDAR INFORMATION
# ============================================================

times = ds.time.values

months = np.array(
    [
        t.astype("datetime64[M]").astype(int) % 12 + 1
        for t in times
    ]
)

years = np.array(
    [
        t.astype("datetime64[Y]").astype(int) + 1970
        for t in times
    ]
)

growing = np.isin(
    months,
    [10, 11, 12, 1, 2, 3, 4]
)

harvest = (
    years
    +
    (months >= 10).astype(int)
)


# ============================================================
# 5. EVENT TRACKING FUNCTION FOR ONE PIXEL
# ============================================================

def build_fd_mask(
    spei1_series,
    spei3_series
):

    n = len(spei1_series)

    mask = np.zeros(
        n,
        dtype=np.int8
    )

    # --------------------------------------------------------
    # 20-day SPEI-1 change
    # --------------------------------------------------------

    change20 = np.full(
        n,
        np.nan,
        dtype=np.float32
    )

    change20[DECLINE_WINDOW:] = (
        spei1_series[DECLINE_WINDOW:]
        -
        spei1_series[:-DECLINE_WINDOW]
    )

    # --------------------------------------------------------
    # Rapid onset condition
    # --------------------------------------------------------

    onset_trigger = (
        (change20 <= RAPID_DECLINE_THRESHOLD)
        &
        (spei1_series <= SPEI1_ONSET_THRESHOLD)
    )

    i = 0

    while i < n:

        # Onset must happen during Oct-Apr
        if not growing[i]:
            i += 1
            continue

        if (
            not onset_trigger[i]
            or np.isnan(spei3_series[i])
        ):
            i += 1
            continue

        event_harvest_year = harvest[i]

        # Only complete harvest years
        if (
            event_harvest_year < 1982
            or event_harvest_year > 2025
        ):
            i += 1
            continue

        # ----------------------------------------------------
        # Confirm SPEI-3 <= -1.0 within 20 days
        # ----------------------------------------------------

        j = i

        while (
            j < n
            and growing[j]
            and harvest[j] == event_harvest_year
        ):

            if np.isnan(spei3_series[j]):
                break

            if (
                spei3_series[j]
                <= SPEI3_DROUGHT_THRESHOLD
            ):
                break

            if (j - i) >= DECLINE_WINDOW:
                break

            j += 1

        # No valid confirmation
        if (
            j >= n
            or np.isnan(spei3_series[j])
            or not growing[j]
            or harvest[j] != event_harvest_year
            or spei3_series[j] > SPEI3_DROUGHT_THRESHOLD
            or (j - i) > DECLINE_WINDOW
        ):

            i += 1
            continue

        # ----------------------------------------------------
        # Event starts here
        # ----------------------------------------------------

        event_start = j

        k = event_start

        # ----------------------------------------------------
        # Track until SPEI-3 > -0.5
        # May continue beyond April
        # ----------------------------------------------------

        while k < n:

            if np.isnan(spei3_series[k]):
                break

            if (
                spei3_series[k]
                > SPEI3_TERMINATION_THRESHOLD
            ):
                break

            k += 1

        # Mark active event days
        mask[event_start:k] = 1

        # Move beyond termination
        i = max(
            k + 1,
            i + 1
        )

    return mask


# ============================================================
# 6. PROCESS LATITUDE BLOCKS
# ============================================================

nlat = ds.sizes["lat"]
nlon = ds.sizes["lon"]

mask_blocks = []

print("\nLatitude rows:", nlat)
print("Block size:", block_size)


for start in range(
    0,
    nlat,
    block_size
):

    end = min(
        start + block_size,
        nlat
    )

    print(
        f"\nProcessing latitude rows "
        f"{start}:{end} of {nlat}"
    )

    spei1_block = (
        ds["spei1"]
        .isel(
            lat=slice(start, end)
        )
        .load()
        .values
        .astype(np.float32)
    )

    spei3_block = (
        ds["spei3"]
        .isel(
            lat=slice(start, end)
        )
        .load()
        .values
        .astype(np.float32)
    )

    block_lat_count = end - start

    mask_block = np.zeros(
        (
            ds.sizes["time"],
            block_lat_count,
            nlon
        ),
        dtype=np.int8
    )

    for ilat in range(
        block_lat_count
    ):

        for ilon in range(
            nlon
        ):

            s1 = spei1_block[
                :,
                ilat,
                ilon
            ]

            s3 = spei3_block[
                :,
                ilat,
                ilon
            ]

            if (
                np.all(np.isnan(s1))
                or
                np.all(np.isnan(s3))
            ):
                continue

            mask_block[
                :,
                ilat,
                ilon
            ] = build_fd_mask(
                s1,
                s3
            )

    mask_da = xr.DataArray(
        mask_block,
        dims=(
            "time",
            "lat",
            "lon"
        ),
        coords={
            "time": ds.time.values,
            "lat": ds.lat.values[start:end],
            "lon": ds.lon.values
        },
        name="flash_drought_active"
    )

    mask_blocks.append(
        mask_da
    )

    del spei1_block
    del spei3_block
    del mask_block

    gc.collect()


# ============================================================
# 7. COMBINE BLOCKS
# ============================================================

print("\nCombining daily mask blocks...")

flash_drought_active = xr.concat(
    mask_blocks,
    dim="lat"
)

flash_drought_active.attrs["long_name"] = (
    "Daily flash drought active mask"
)

flash_drought_active.attrs["description"] = (
    "1 = active flash drought event, "
    "0 = no active flash drought"
)


# ============================================================
# 8. CHECK RESULT
# ============================================================

print("\n" + "=" * 70)
print("FLASH DROUGHT DAILY MASK CHECK")
print("=" * 70)

print(
    "Shape:",
    flash_drought_active.shape
)

print(
    "First date:",
    flash_drought_active.time.values[0]
)

print(
    "Last date:",
    flash_drought_active.time.values[-1]
)

print(
    "Active drought days:",
    int(
        flash_drought_active.sum().values
    )
)

print(
    "Maximum value:",
    int(
        flash_drought_active.max().values
    )
)

print(
    "Minimum value:",
    int(
        flash_drought_active.min().values
    )
)


# ============================================================
# 9. TEST LOCATION
# ============================================================

test = flash_drought_active.sel(
    lat=-27.95,
    lon=21.05,
    method="nearest"
)

print("\nTest location")
print("-" * 40)

print(
    "Total active days:",
    int(test.sum().values)
)

print(
    "First 20 values:",
    test.values[:20]
)


# ============================================================
# 10. SAVE
# ============================================================

flash_drought_active.to_dataset().to_netcdf(
    output_file
)

print("\nSaved successfully:")
print(output_file)

print("\nDONE!")

DAILY FLASH DROUGHT ACTIVE MASK

Dimensions:
{'time': 16436, 'lat': 141, 'lon': 220}

First date: 1981-01-01T00:00:00.000000000
Last date : 2025-12-31T00:00:00.000000000

Latitude rows: 141
Block size: 3

Processing latitude rows 0:3 of 141

Processing latitude rows 3:6 of 141

Processing latitude rows 6:9 of 141

Processing latitude rows 9:12 of 141

Processing latitude rows 12:15 of 141

Processing latitude rows 15:18 of 141

Processing latitude rows 18:21 of 141

Processing latitude rows 21:24 of 141

Processing latitude rows 24:27 of 141

Processing latitude rows 27:30 of 141

Processing latitude rows 30:33 of 141

Processing latitude rows 33:36 of 141

Processing latitude rows 36:39 of 141

Processing latitude rows 39:42 of 141

Processing latitude rows 42:45 of 141

Processing latitude rows 45:48 of 141

Processing latitude rows 48:51 of 141

Processing latitude rows 51:54 of 141

Processing latitude rows 54:57 of 141

Processing latitude rows 57:60 of 141

Processing latitude ro

In [26]:
import xarray as xr
import os

heatwave_file = r"C:\Users\franc\south_africa_maize_season_heatwaves_1981_2025.nc"

print("File exists:", os.path.exists(heatwave_file))

hw = xr.open_dataset(heatwave_file)

print("\nVariables:")
print(list(hw.data_vars))

print("\nDimensions:")
print(dict(hw.sizes))

print("\nCoordinates:")
print(list(hw.coords))

print("\nVariable details:")
for var in hw.data_vars:
    print(
        var,
        "dims =", hw[var].dims,
        "shape =", hw[var].shape,
        "dtype =", hw[var].dtype
    )

File exists: True

Variables:
['hw_frequency', 'hw_duration', 'hw_intensity']

Dimensions:
{'latitude': 57, 'longitude': 89, 'harvest_year': 46}

Coordinates:
['number', 'latitude', 'longitude', 'harvest_year']

Variable details:
hw_frequency dims = ('latitude', 'longitude', 'harvest_year') shape = (57, 89, 46) dtype = int64
hw_duration dims = ('latitude', 'longitude', 'harvest_year') shape = (57, 89, 46) dtype = int64
hw_intensity dims = ('harvest_year', 'latitude', 'longitude') shape = (46, 57, 89) dtype = float64


In [27]:
print("\nCoordinate ranges:")

for coord in hw.coords:
    if coord in ["time", "lat", "lon", "latitude", "longitude", "harvest_year"]:
        print(coord)

        if coord == "time":
            print(" First:", hw[coord].values[0])
            print(" Last :", hw[coord].values[-1])

        elif hw[coord].ndim == 1:
            print(" First:", hw[coord].values[0])
            print(" Last :", hw[coord].values[-1])


Coordinate ranges:
latitude
 First: -21.0
 Last : -35.0
longitude
 First: 11.0
 Last : 33.0
harvest_year
 First: 1981
 Last : 2026
